# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Dataset Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset on second primary colorectal cancer in cancer survivors using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant


## 1. Data Loading
Load dataset metadata and get an overall description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets and their fields using their `@id` attributes.

In [ ]:
# Get all record sets with their @id
print("Available record sets (by @id):")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs['name']}")

# Display their fields by @id
print("\nFields in each record set:")
for rs in record_sets:
    print(f"\nRecord set: {rs['name']} (ID: {rs['@id']})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # Only one field
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"  - {field.get('@id', '[no id]')}: {field.get('name', '[no name]')}")
        else:
            print(f"  - {field}")

## 3. Data Extraction
Load records from a selected record set (using its `@id`) into a pandas DataFrame. The @id values above are used directly as arguments.

In [ ]:
# Extract data from each record set as DataFrames
record_set_ids = [rs['@id'] for rs in record_sets]  # All available record sets by @id
dfs = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(records)

# Pick first record set and display its fields and the first rows
selected_record_set_id = record_set_ids[0] if record_set_ids else None

if selected_record_set_id:
    print(f"Extracted columns for record set {selected_record_set_id}:")
    print(dfs[selected_record_set_id].columns.tolist())
    display(dfs[selected_record_set_id].head())
else:
    print("No record sets were found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Let's analyze the data by filtering and normalizing a numeric field, and grouping by another variable. All fields are referenced by their `@id` attributes as shown above.

In [ ]:
# Example: If the dataset has a numeric field (e.g., age), filter and normalize

# Inspect the available columns
if selected_record_set_id and not dfs[selected_record_set_id].empty:
    df = dfs[selected_record_set_id]
    print("Columns in the main record set:", df.columns.tolist())
    # Try to find a numeric field (by naive type check or by typical names)
    numeric_field_id = None
    for col in df.columns:
        # Try to find an 'age' or another numeric-looking column
        if 'age' in col.lower():
            numeric_field_id = col
            break
    # If none found, try first numeric column by dtype
    if numeric_field_id is None:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    if numeric_field_id is not None:
        print(f"\nUsing numeric field '@id': {numeric_field_id}")
        # Replace missing or non-numeric with NaN
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()  # Use mean as threshold example
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        col_norm = f"{numeric_field_id}_normalized"
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, col_norm]].head())

        # Try to group by a categorical field (e.g., 'sex' field)
        group_field = None
        for col in df.columns:
            if 'sex' in col.lower() or 'gender' in col.lower():
                group_field = col
                break
        # If not, pick first object/categorical column not used
        if group_field is None:
            for col in df.columns:
                if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                    group_field = col
                    break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped summary of {numeric_field_id} by {group_field}:")
            display(grouped_df)
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No suitable numeric field found for demonstration.")
else:
    print("Dataset is empty or not loaded.")

## 5. Visualization
Visualize the distribution of the selected numeric field (e.g., age) and its grouping if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and not dfs[selected_record_set_id].empty and 'numeric_field_id' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
This notebook illustrated step-by-step usage of the `mlcroissant` library to explore a structured, FAIR dataset via its Croissant schema. We loaded the dataset, inspected its record sets and fields by `@id`, extracted data into DataFrames, and performed simple EDA and visualization. Further analyses specific to clinical or biomarker research can be done by referencing field and column `@id`s as above.